# Escenas paramétricas: Creación de objetos desde datos

## Generación de coordenadas para definir puntos en el espacio 3D

In [7]:
import numpy as np
import trimesh
!pip install open3d
import open3d as o3d

Defaulting to user installation because normal site-packages is not writeable


ERROR: Could not find a version that satisfies the requirement open3d (from versions: none)
ERROR: No matching distribution found for open3d


ModuleNotFoundError: No module named 'open3d'

In [8]:
import sys
print(sys.version)

3.13.1 (tags/v3.13.1:0671451, Dec  3 2024, 19:06:28) [MSC v.1942 64 bit (AMD64)]


In [3]:
points = []
for x in range(-2, 3):
    for y in range(-2, 3):
        for z in range(-2, 3):
            points.append([x, y, z])

points = np.array(points)
print(points[:5])

[[-2 -2 -2]
 [-2 -2 -1]
 [-2 -2  0]
 [-2 -2  1]
 [-2 -2  2]]


## Generación de primitivas en trimesh

In [4]:
meshes = []

for p in points:
    x, y, z = p

    # Condición para elegir forma
    if (x + y + z) % 3 == 0:
        mesh = trimesh.creation.box(extents=(0.5, 0.5, 0.5))
    elif (x + y + z) % 3 == 1:
        mesh = trimesh.creation.icosphere(radius=0.3)
    else:
        mesh = trimesh.creation.cylinder(radius=0.2, height=0.6)

    # Mover la figura a la posición del punto
    mesh.apply_translation(p)

    # Cambiar color dinámicamente
    color = [abs(x)*40, abs(y)*40, abs(z)*40, 255]
    mesh.visual.face_colors = color

    meshes.append(mesh)

# Combinar todo en una sola escena
scene = trimesh.util.concatenate(meshes)

## exportaciones

In [5]:
scene.export('../media/scene_trimesh.obj')
scene.export('../media/scene_trimesh.stl')
scene.export('../media/scene_trimesh.glb')  # GLTF binario

b'glTF\x02\x00\x00\x00\x04F\x12\x00\xd8\x03\x00\x00JSON{"scene":0,"scenes":[{"nodes":[0]}],"asset":{"version":"2.0","generator":"https://github.com/mikedh/trimesh"},"accessors":[{"componentType":5125,"type":"SCALAR","bufferView":0,"count":178884,"max":[30063],"min":[0]},{"componentType":5126,"type":"VEC3","byteOffset":0,"bufferView":1,"count":30064,"max":[2.299999952316284,2.299999952316284,2.299999952316284],"min":[-2.299999952316284,-2.299999952316284,-2.299999952316284]},{"componentType":5121,"normalized":true,"type":"VEC4","byteOffset":0,"bufferView":2,"count":30064,"max":[80,80,80,255],"min":[0,0,0,255]}],"meshes":[{"name":"geometry_0","extras":{"shape":"radius"},"primitives":[{"attributes":{"POSITION":1,"COLOR_0":2},"indices":0,"mode":4}]}],"nodes":[{"name":"world","children":[1]},{"name":"geometry_0","mesh":0}],"buffers":[{"byteLength":1196560}],"bufferViews":[{"buffer":0,"byteOffset":0,"byteLength":715536},{"buffer":0,"byteOffset":715536,"byteLength":360768},{"buffer":0,"byteOf

In [ ]:

# Convertir trimesh → open3d
vertices = np.array(scene.vertices)
triangles = np.array(scene.faces)

mesh_o3d = o3d.geometry.TriangleMesh()
mesh_o3d.vertices = o3d.utility.Vector3dVector(vertices)
mesh_o3d.triangles = o3d.utility.Vector3iVector(triangles)

mesh_o3d.compute_vertex_normals()

# Exportar
o3d.io.write_triangle_mesh("scene_open3d.ply", mesh_o3d)
o3d.io.write_triangle_mesh("scene_open3d.stl", mesh_o3d)